### 1. Download GoEmotions dataset

In [ ]:
# install packages (only for the first time)
!pip install datasets pandas
!pip install -U datasets huggingface_hub fsspec

from datasets import Dataset, load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import GPT2Tokenizer, GPT2LMHeadModel, TextDataset, Trainer, TrainingArguments, DataCollatorForLanguageModeling, pipeline
import torch
import os
os.environ["WANDB_DISABLED"] = "true"

# load GoEmotions dataset
goemotions = load_dataset("go_emotions", "simplified", split="train")

# convert it into a pandas DataFrame Pandas
df = goemotions.to_pandas()

# Keep samples with only one label
df["num_labels"] = df["labels"].apply(len)
df_single_label = df[df["num_labels"] == 1].copy()

# map label integers with emotions names
label_names = goemotions.features["labels"].feature.names
df_single_label["mood"] = df_single_label["labels"].apply(lambda x: label_names[x[0]])
df_single_label["caption"] = df_single_label["text"]

# select relevant features
final_df = df_single_label[["mood", "caption"]]

# save on csv file
final_df.to_csv("mood_captions_goemotions.csv", index=False)

# print some sample rows
print(final_df.sample(10))


  Using cached fsspec-2025.7.0-py3-none-any.whl.metadata (12 kB)
              mood                                            caption
37634      neutral                    Shout out to the two messy bots
514      gratitude  My coffee burned its way through my nostrils. ...
25861       relief  My god, what is WRONG with people? So glad you...
27654      neutral                  You have been rewarded my friend.
11228     approval  Let's be fair to them and say not forever. But...
40559  disapproval                                      Please don't.
33139      neutral                                  Probably therapy.
26319      neutral  Crows get hungry too you know and it's not lik...
38666  disapproval  I got an unsolicited text from wal-mart earlie...
18334         love  I dunno, I love me some barrel-aged spirits an...


### 2. Load and split dataset

In [ ]:
# load the previously created dataset from csv file
df = pd.read_csv("mood_captions_goemotions.csv")

# Crea un prompt testuale che combina mood e caption
# Il modello imparerà a generare il testo che segue "CAPTION:"
df['text'] = df.apply(lambda row: f"MOOD: {row['mood']}\nCAPTION: {row['caption']}", axis=1)

# Dividi il dataset in training e validation
train_df, val_df = train_test_split(df, test_size=0.1)

# Salva i set di dati in file di testo
with open('train.txt', 'w') as f:
    f.write('\n'.join(train_df['text']))

with open('val.txt', 'w') as f:
    f.write('\n'.join(val_df['text']))


### 3. Initialize Model and Tokenizer

In [ ]:
# Scegli il modello (es. 'gpt2' o 'distilgpt2')
model_name = 'distilgpt2'

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

# Aggiungi un pad token se non è presente (utile per il batching)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})
    model.resize_token_embeddings(len(tokenizer))

### 4. Create datasets for training

In [ ]:
# Create datasets for training
train_dataset = TextDataset(
    tokenizer=tokenizer,
    file_path='train.txt',
    block_size=128  # Lunghezza massima delle sequenze
)

val_dataset = TextDataset(
    tokenizer=tokenizer,
    file_path='val.txt',
    block_size=128
)

# Data collator regourps samples in batches
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

/usr/local/lib/python3.11/dist-packages/transformers/data/datasets/language_modeling.py:53: FutureWarning: This dataset will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/language-modeling/run_mlm.py
  warnings.warn(


### 5. Model Fine-tuning

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',          # Folder for saving the model
    overwrite_output_dir=True,
    num_train_epochs=20,             # Number of epochs for training
    per_device_train_batch_size=8,   # Batch size for GPU/CPU
    per_device_eval_batch_size=8,
    eval_steps=500,                  # Evaluation frequency
    save_steps=1000,                 # Model saving frequency
    warmup_steps=500,                # Number of warmup steps
    prediction_loss_only=True,
    logging_dir='./logs',
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

# Start fine-tuning
trainer.train()

# Save final model
trainer.save_model("./modello_caption_divertenti")
tokenizer.save_pretrained("./modello_caption_divertenti")

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Step,Training Loss
500,2.941600
1000,2.927900
1500,2.848300
2000,2.765000
2500,2.746200
3000,2.670100
3500,2.638800
4000,2.616200
4500,2.560500
5000,2.538500


('./modello_caption_divertenti/tokenizer_config.json',
 './modello_caption_divertenti/special_tokens_map.json',
 './modello_caption_divertenti/vocab.json',
 './modello_caption_divertenti/merges.txt',
 './modello_caption_divertenti/added_tokens.json')

### 6. Generating captions

In [ ]:
# Load the fine-tuned model
generator = pipeline('text-generation', model='./modello_caption_divertenti', tokenizer='./modello_caption_divertenti')

# define input mooed
mood = "concerned"
prompt = f"MOOD: {mood}\nCAPTION:"

# Generate caption
generated_text = generator(prompt, max_length=50, num_return_sequences=1)

# Extract and clean generated caption
caption = generated_text[0]['generated_text'].split('CAPTION:')[1].strip()

print(f"Mood: {mood}")
print(f"Generated Caption: {caption}")

Device set to use cuda:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Mood: concerned
Generated Caption: [NAME] is a fucking moron
MOOD: neutral


In [ ]:
# define input mooed
mood = "happy"
prompt = f"MOOD: {mood}\nCAPTION:"

# Generate caption
generated_text = generator(prompt, max_length=50, num_return_sequences=1)

# Extract and clean generated caption
caption = generated_text[0]['generated_text'].split('CAPTION:')[1].strip()

print(f"Mood: {mood}")
print(f"Generated Caption: {caption}")

Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Mood: happy
Generated Caption: You're welcome, man. Have a nice day. 
MOOD: neutral


In [ ]:
# define input mooed
mood = "sad"
prompt = f"MOOD: {mood}\nCAPTION:"

# Generate caption
generated_text = generator(prompt, max_length=50, num_return_sequences=1)

# Extract and clean generated caption
caption = generated_text[0]['generated_text'].split('CAPTION:')[1].strip()

print(f"Mood: {mood}")
print(f"Generated Caption: {caption}")

Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Mood: sad
Generated Caption: Yeah I’m too late to watch this movie. I’ll watch it once it’s finished.
MOOD: gratitude


In [ ]:
# define input mooed
mood = "fear"
prompt = f"MOOD: {mood}\nCAPTION:"

# Generate caption
generated_text = generator(prompt, max_length=50, num_return_sequences=1)

# Extract and clean generated caption
caption = generated_text[0]['generated_text'].split('CAPTION:')[1].strip()

print(f"Mood: {mood}")
print(f"Generated Caption: {caption}")

Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Mood: fear
Generated Caption: I'm afraid you're too afraid. I'm glad you're not alone.
MOOD: neutral
